# Day 07 - 10 Minute Map | A World of Forests 🌳

How much of each country’s land area is covered by forests?

For Day 07 of the #30DayMapChallenge 2026, I adapted Milan Janosov’s coffee mapping workflow to map forest area by country in 2024.

## Tools

Python · pandas · GeoPandas · Matplotlib

## Workflow

1. Read the 2024 forest area figures from the World Bank CSV.
2. Join the figures to Natural Earth boundaries using country codes.
3. Plot the map in the Robinson projection and export it as a square PNG.

## Local data

- `API_AG.LND.FRST.ZS_DS2_en_csv_v2_811.csv`: forest area as a percentage of land area.
- `countries.zip`: Natural Earth boundaries, downloaded by the notebook.

In [ ]:
import pandas as pd

forest = pd.read_csv("API_AG.LND.FRST.ZS_DS2_en_csv_v2_811.csv", skiprows=4)
forest[["Country Name", "Country Code", "2024"]].head()

In [ ]:
from urllib.request import urlretrieve
import geopandas as gpd

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
urlretrieve(url, "countries.zip")

world = gpd.read_file("zip://countries.zip")
world = world.loc[world["ADMIN"] != "Antarctica"].copy()

forest_2024 = forest.set_index("Country Code")["2024"]
world["forest_pct"] = world["WB_A3"].map(forest_2024).fillna(
    world["ISO_A3_EH"].map(forest_2024)
)

print(f"Countries and territories with data: {world['forest_pct'].notna().sum()}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle
from pyproj import Geod, Transformer

paper = "#f9f6ef"
ocean = "#edf3f1"
ink = "#263c30"
no_data = "#b7b3a8"

cmap = LinearSegmentedColormap.from_list(
    "forest", ["#e9d7a8", "#a6b777", "#5b8050", "#2e5c40", "#123827"]
)
norm = Normalize(0, 100)
crs = "ESRI:54030"
mapped = world.to_crs(crs)

# Fit the map to the available width without stretching it
page_w, page_h = 300, 195
fig = plt.figure(figsize=(page_w / 25.4, page_h / 25.4), facecolor=paper)

xmin, ymin, xmax, ymax = mapped.total_bounds
xmin += (xmax - xmin) * 0.07  # Crop 7% from the left edge
dx, dy = xmax - xmin, ymax - ymin
xpad, ypad = dx * 0.006, dy * 0.012

map_left, map_width, map_top = 0.024, 0.952, 0.87
map_height = (
    map_width * page_w / page_h * (dy + 2 * ypad) / (dx + 2 * xpad)
)
map_bottom = map_top - map_height

# Draw the frame and country polygons
fig.add_artist(Rectangle(
    (0.022, 0.087), 0.956, 0.798, transform=fig.transFigure,
    facecolor=ocean, edgecolor=ink, linewidth=0.9, zorder=0,
))

ax = fig.add_axes([map_left, map_bottom, map_width, map_height])
mapped.plot(
    ax=ax, column="forest_pct", cmap=cmap, norm=norm,
    edgecolor=ink, linewidth=0.22,
    missing_kwds={"color": no_data, "edgecolor": ink, "linewidth": 0.22},
)
ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)
ax.set_aspect("equal")
ax.set_axis_off()

fig.text(0.5, 0.937, "Day 07 | A World of Forests",
         ha="center", va="center", fontsize=29,
         family="serif", weight="bold", color=ink)

# North arrow
fig.text(0.946, 0.847, "N", ha="center", va="center",
         fontsize=10, weight="bold", color=ink)
ax.annotate(
    "", xy=(0.946, 0.828), xytext=(0.946, 0.786),
    xycoords=fig.transFigure, annotation_clip=False,
    arrowprops={"arrowstyle": "-|>", "color": ink,
                "lw": 1.1, "mutation_scale": 13},
)

# Compact forest area legend
legend_y = map_bottom - 0.065
fig.text(0.077, legend_y + 0.029, "Forest area (% of land area) · 2024",
         fontsize=9, color=ink)

bar_ax = fig.add_axes([0.077, legend_y, 0.31, 0.012])
bar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    cax=bar_ax, orientation="horizontal",
    ticks=[0, 25, 50, 75, 100],
)
bar.ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
bar.ax.tick_params(labelsize=8, colors=ink, length=2, pad=2)
bar.outline.set_visible(False)

fig.add_artist(Rectangle(
    (0.43, legend_y), 0.011, 0.015, transform=fig.transFigure,
    facecolor=no_data, edgecolor=ink, linewidth=0.5,
))
fig.text(0.448, legend_y + 0.0075, "No data",
         va="center", fontsize=8, color=ink)

# Scale bar calibrated at the equator
geod = Geod(ellps="WGS84")
project = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
lon_end, lat_end, _ = geod.fwd(0, 0, 90, 5_000_000)
x0, _ = project.transform(0, 0)
x1, _ = project.transform(lon_end, lat_end)
scale_width = map_width * abs(x1 - x0) / (dx + 2 * xpad)

sx, sy = 0.92 - scale_width, legend_y
for i, fill in enumerate([ink, paper]):
    fig.add_artist(Rectangle(
        (sx + i * scale_width / 2, sy), scale_width / 2, 0.009,
        transform=fig.transFigure, facecolor=fill,
        edgecolor=ink, linewidth=0.6,
    ))

for fraction, label in [(0, "0"), (0.5, "2,500"), (1, "5,000 km")]:
    fig.text(sx + fraction * scale_width, sy + 0.014, label,
             ha="center", va="bottom", fontsize=8, color=ink)

fig.text(sx + scale_width / 2, sy - 0.011, "Scale at the equator",
         ha="center", va="top", fontsize=7.5, color=ink)

# Map credit and export
fig.text(
    0.5, 0.043,
    "#30DayMapChallenge | 2026 | Day 07 | 10 Minute Map | "
    "A World of Forests | Map by María Victoria León Parra",
    ha="center", va="center", fontsize=8.5, family="serif", color=ink,
)

fig.savefig("Day07_A_World_of_Forests.png", dpi=300)
plt.show()

## Sources and inspiration

- [Forest area (% of land area), FAO via World Bank](https://data.worldbank.org/indicator/AG.LND.FRST.ZS)
- [Natural Earth country boundaries](https://www.naturalearthdata.com/downloads/110m-cultural-vectors/110m-admin-0-countries/)
- [Mapping Global Coffee in 10 Minutes by Milan Janosov](https://www.youtube.com/watch?v=a3CB5x3XQmg)

## Notes

The join matched 168 countries and territories. Grey indicates no data. The map shows the percentage of land covered by forests in each country, rather than the location of individual forests.

## Author

Map by María Victoria León Parra

[Portfolio](https://vl-portfolio.lovable.app/portfolio) · [Back to the challenge](https://github.com/mvlparra-lab/30DayMapChallenge-2026)